# 05 · Representación por bloques, anomalías y validación agrupada

Usa los artefactos full-cohort del 03 y 04. La representación biológica se construye por
bloques —semi-cuantificación, primer orden, forma y textura— después de ajustar cada feature
dentro de familia de adquisición. Las variables técnicas se reservan para QC, dominio y una
auditoría explícita de shortcuts.

> PCA, t-SNE y anomalías son herramientas de revisión. No son predictores finales, subtipos
> clínicos ni probabilidades de patología.

## 1. Carga, linaje y separación de roles

El notebook exige cobertura full-cohort. No combina percentiles crudos de intensidad con
biomarcadores. Los casos con flags de registro se conservan y se muestran por separado.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import sys
import time
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, clear_output, display
from sklearn.covariance import LedoitWolf
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.manifold import TSNE
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Importar desde los módulos concretos evita depender de que un kernel
# Jupyter conserve una versión antigua de modeling.__init__ en memoria.
from modeling.embedding_search import run_embedding_search
from modeling.optuna_models import FoldData, run_optuna_experiment

PRIVATE_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'private_eda'
NODE4_PROFILE = 'v3'
NODE4_OUTPUT_DIR = PRIVATE_OUTPUT_DIR / f'full_cohort_{NODE4_PROFILE}'
NODE5_RUN_ID = os.environ.get('DAT_NODE5_RUN_ID', 'optuna_radiomics_v1')
if not NODE5_RUN_ID or any(
    character not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-._'
    for character in NODE5_RUN_ID
):
    raise ValueError('DAT_NODE5_RUN_ID sólo admite letras, números, punto, guion y _.')
NODE5_OUTPUT_DIR = PRIVATE_OUTPUT_DIR / 'node5_runs' / NODE5_RUN_ID
EMBEDDING_RUN_ID = os.environ.get('DAT_EMBEDDING_RUN_ID', 'umap_tsne_v1')
if not EMBEDDING_RUN_ID or any(
    character not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-._'
    for character in EMBEDDING_RUN_ID
):
    raise ValueError(
        'DAT_EMBEDDING_RUN_ID sólo admite letras, números, punto, guion y _.'
    )
EMBEDDING_SEARCH_DIR = (
    NODE5_OUTPUT_DIR / 'embedding_runs' / EMBEDDING_RUN_ID
)
QC_PATH = PRIVATE_OUTPUT_DIR / 'image_qc_manifest.csv'
FEATURES_PATH = NODE4_OUTPUT_DIR / 'dat_radiomics_features_full.csv'
REGISTRATION_PATH = NODE4_OUTPUT_DIR / 'registration_qc_full.csv'
FAILURES_PATH = NODE4_OUTPUT_DIR / 'registration_failures_full.csv'
UPSTREAM_CONFIG_PATH = NODE4_OUTPUT_DIR / 'registration_radiomics_full_config.json'
CROPS_DIR = NODE4_OUTPUT_DIR / 'registered_crops'
NODE5_CONFIG_PATH = NODE5_OUTPUT_DIR / 'node5_analysis_config.json'
COVERAGE_PATH = NODE5_OUTPUT_DIR / 'node5_coverage_qc_full.csv'
EMBEDDING_PATH = NODE5_OUTPUT_DIR / 'embedding_coordinates_full.csv'
OUTLIER_PATH = NODE5_OUTPUT_DIR / 'outlier_scores_full.csv'
OPTUNA_OUTPUT_DIR = NODE5_OUTPUT_DIR / 'optuna_models'
OOF_PATH = OPTUNA_OUTPUT_DIR / 'optuna_oof_predictions.csv'
METRICS_PATH = OPTUNA_OUTPUT_DIR / 'optuna_oof_metrics.csv'
LABEL_AUDIT_PATH = NODE5_OUTPUT_DIR / 'label_audit_full.csv'
PROTOCOL_DIAGNOSTICS_PATH = NODE5_OUTPUT_DIR / 'protocol_diagnostics_full.csv'
OPTUNA_MODULE_PATH = PROJECT_ROOT / 'modeling' / 'optuna_models.py'

RANDOM_SEED = 20260821
MAX_MISSING_FRACTION = 0.20
CORRELATION_THRESHOLD = 0.95
MAX_EMBEDDING_ROWS = 1500
NEIGHBORS_TO_SHOW = 5
CPU_JOBS = max(1, (os.cpu_count() or 2) - 1)
ISOLATION_TREES = 500
TSNE_MAX_ITER = 1500
MIN_FEATURE_COVERAGE = 0.995
MAX_FAMILY_FAILURE_RATE = 0.10
MIN_FAMILY_GATE_SIZE = 10
MIN_BACKGROUND_VALID_FRACTION = 0.80
OPTUNA_TRIALS_PER_MODEL = int(os.environ.get('DAT_OPTUNA_TRIALS', '50'))
EMBEDDING_TRIALS_PER_METHOD = int(
    os.environ.get('DAT_EMBEDDING_TRIALS', '30')
)
EMBEDDING_SEEDS = (RANDOM_SEED, RANDOM_SEED + 1, RANDOM_SEED + 2)
OPTUNA_MODELS = ('logistic', 'random_forest', 'xgboost')
OPTUNA_FEATURE_SET = 'radiomics_only'
PREFER_XGBOOST_GPU = True
RESUME = True
SAVE_PRIVATE_OUTPUTS = True

for required_path in [
    QC_PATH, FEATURES_PATH, REGISTRATION_PATH, FAILURES_PATH,
    UPSTREAM_CONFIG_PATH, CROPS_DIR,
]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

NODE5_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid')
display(Markdown(
    f'**Upstream nodo 04:** `{NODE4_PROFILE}` (sólo lectura) · '
    f'**experimento nodo 05:** `{NODE5_RUN_ID}` · '
    f'**Optuna:** `{OPTUNA_TRIALS_PER_MODEL}` ensayos completos/modelo · '
    f'**embeddings:** `{EMBEDDING_RUN_ID}`, '
    f'`{EMBEDDING_TRIALS_PER_METHOD}` ensayos/método · '
    f'**CPU:** `{CPU_JOBS}` hilos · **GPU:** XGBoost CUDA con fallback CPU.'
))

In [ ]:
qc = pd.read_csv(QC_PATH, dtype={'uid': 'string'})
features = pd.read_csv(FEATURES_PATH, dtype={'uid': 'string'})
registration = pd.read_csv(REGISTRATION_PATH, dtype={'uid': 'string'})
failures = pd.read_csv(FAILURES_PATH, dtype={'uid': 'string'})
upstream_config = json.loads(UPSTREAM_CONFIG_PATH.read_text(encoding='utf-8'))


def replace_with_retry(
    temporary: Path, destination: Path, attempts: int = 10
) -> None:
    delay_seconds = 0.25
    for attempt in range(attempts):
        try:
            temporary.replace(destination)
            return
        except PermissionError:
            if attempt == attempts - 1:
                raise
            time.sleep(delay_seconds)
            delay_seconds = min(delay_seconds * 1.8, 3.0)


def atomic_csv(frame: pd.DataFrame, destination: Path) -> None:
    temporary = destination.with_suffix('.csv.tmp')
    frame.to_csv(temporary, index=False)
    replace_with_retry(temporary, destination)


node5_config = {
    'algorithm_version': 'node5_optuna_radiomics_v1',
    'node4_profile': NODE4_PROFILE,
    'node5_run_id': NODE5_RUN_ID,
    'upstream_config_hash': upstream_config.get('config_hash'),
    'optuna_module_hash': hashlib.sha256(
        OPTUNA_MODULE_PATH.read_bytes()
    ).hexdigest(),
    'cohort_uid_hash': hashlib.sha256(
        '|'.join(qc['uid'].astype(str)).encode('utf-8')
    ).hexdigest(),
    'max_missing_fraction': MAX_MISSING_FRACTION,
    'correlation_threshold': CORRELATION_THRESHOLD,
    'max_embedding_rows': MAX_EMBEDDING_ROWS,
    'isolation_trees': ISOLATION_TREES,
    'tsne_max_iter': TSNE_MAX_ITER,
    'min_feature_coverage': MIN_FEATURE_COVERAGE,
    'max_family_failure_rate': MAX_FAMILY_FAILURE_RATE,
    'min_family_gate_size': MIN_FAMILY_GATE_SIZE,
    'min_background_valid_fraction': MIN_BACKGROUND_VALID_FRACTION,
    'optuna_trials_per_model': OPTUNA_TRIALS_PER_MODEL,
    'optuna_models': OPTUNA_MODELS,
    'optuna_feature_set': OPTUNA_FEATURE_SET,
    'prefer_xgboost_gpu': PREFER_XGBOOST_GPU,
    'random_seed': RANDOM_SEED,
}
node5_hash = hashlib.sha256(
    json.dumps(node5_config, sort_keys=True).encode('utf-8')
).hexdigest()
node5_config['config_hash'] = node5_hash
if NODE5_CONFIG_PATH.exists() and RESUME:
    previous_node5_config = json.loads(
        NODE5_CONFIG_PATH.read_text(encoding='utf-8')
    )
    if previous_node5_config.get('config_hash') != node5_hash:
        raise RuntimeError(
            f'El experimento del nodo 5 `{NODE5_RUN_ID}` pertenece a otra '
            'configuración. Usa un DAT_NODE5_RUN_ID nuevo; no es necesario '
            'mover ni volver a ejecutar el nodo 04.'
        )
else:
    temporary_config = NODE5_CONFIG_PATH.with_suffix('.json.tmp')
    temporary_config.write_text(
        json.dumps(node5_config, indent=2), encoding='utf-8'
    )
    replace_with_retry(temporary_config, NODE5_CONFIG_PATH)

features_for_merge = features.drop(
    columns=[column for column in ['is_pathologic', 'acquisition_family']
             if column in features.columns]
)
registration_for_merge = registration.drop(
    columns=[column for column in ['is_pathologic', 'acquisition_family']
             if column in registration.columns]
)
data = (
    qc.merge(features_for_merge, on='uid', how='left', validate='one_to_one')
    .merge(registration_for_merge, on='uid', how='left', validate='one_to_one')
)
if data['is_pathologic'].isna().any():
    raise ValueError('Hay exámenes sin etiqueta en la cohorte combinada.')
data['is_pathologic'] = data['is_pathologic'].astype(int)
data['acquisition_family'] = data['acquisition_family'].astype(str)
data['feature_available'] = data['uid'].isin(features['uid'])
data['registration_available'] = data['uid'].isin(registration['uid'])
if 'background_qc_valid' not in data:
    raise ValueError('Falta background_qc_valid: vuelve a ejecutar el nodo 04 v3.')
data['background_qc_valid'] = data['background_qc_valid'].fillna(False).astype(bool)
data['background_floor_applied'] = data[
    'background_floor_applied'
].fillna(False).astype(bool)

coverage_by_family = (
    data.groupby('acquisition_family', dropna=False)
    .agg(
        n=('uid', 'size'),
        features=('feature_available', 'sum'),
        registrations=('registration_available', 'sum'),
        background_valid=('background_qc_valid', 'sum'),
    )
    .reset_index()
)
coverage_by_family['failures'] = (
    coverage_by_family['n'] - coverage_by_family['features']
)
coverage_by_family['feature_coverage'] = (
    coverage_by_family['features'] / coverage_by_family['n']
)
coverage_by_family['failure_rate'] = (
    coverage_by_family['failures'] / coverage_by_family['n']
)
coverage_by_family['background_valid_fraction'] = np.divide(
    coverage_by_family['background_valid'],
    coverage_by_family['features'],
    out=np.zeros(len(coverage_by_family), dtype=float),
    where=coverage_by_family['features'].to_numpy() > 0,
)
atomic_csv(coverage_by_family, COVERAGE_PATH)

feature_coverage = float(data['feature_available'].mean())
background_valid_fraction = float(
    data.loc[data['feature_available'], 'background_qc_valid'].mean()
)
gated_families = coverage_by_family[
    coverage_by_family['n'] >= MIN_FAMILY_GATE_SIZE
]
worst_family_failure = float(
    gated_families['failure_rate'].max() if len(gated_families) else 0
)

coverage = pd.DataFrame({
    'indicador': [
        'QC notebook 03', 'Features notebook 04', 'Fallos nodo 04',
        'Cobertura de features', 'Fondos QC válidos',
        'Peor tasa de fallo familiar (n>=gate)',
    ],
    'valor': [
        len(qc), len(features), len(failures), feature_coverage,
        background_valid_fraction, worst_family_failure,
    ],
})
display(coverage.style.hide(axis='index'))
display(Markdown('### Cobertura por familia de adquisición'))
display(
    coverage_by_family.sort_values(
        ['failure_rate', 'n'], ascending=[False, False]
    ).head(30).style.hide(axis='index')
)
if feature_coverage < MIN_FEATURE_COVERAGE:
    raise RuntimeError(
        f'Cobertura de features {feature_coverage:.1%} < '
        f'{MIN_FEATURE_COVERAGE:.1%}. Corrige/reanuda el nodo 04 antes de modelar.'
    )
if worst_family_failure > MAX_FAMILY_FAILURE_RATE:
    raise RuntimeError(
        f'Tasa máxima de fallo familiar {worst_family_failure:.1%} > '
        f'{MAX_FAMILY_FAILURE_RATE:.1%}. El subconjunto no es representativo.'
    )

biological_blocks = {
    'semiquant': [column for column in data if column.startswith('semiquant_')],
    'firstorder': [column for column in data if column.startswith('firstorder_ratio_')],
    'shape': [column for column in data if column.startswith('shape_active_')],
    'texture': [column for column in data if column.startswith('texture3d_')],
}
background_blocks_enabled = (
    background_valid_fraction >= MIN_BACKGROUND_VALID_FRACTION
)
if not background_blocks_enabled:
    biological_blocks['semiquant'] = []
    biological_blocks['firstorder'] = []
    display(Markdown(
        '**Gate de fondo:** se excluyen semiquant y first-order porque sólo '
        f'`{background_valid_fraction:.1%}` tiene fondo QC válido.'
    ))
ratio_columns = [
    column
    for block in ['semiquant', 'firstorder']
    for column in biological_blocks[block]
]
for column in ratio_columns:
    values = pd.to_numeric(data[column], errors='coerce')
    data[column] = np.sign(values) * np.log1p(np.abs(values))
stability_columns = [
    column for column in data if column.startswith('stability_')
]
qc_source_columns = set(qc.select_dtypes(include=np.number).columns)
registration_numeric = set(
    registration_for_merge.select_dtypes(include=np.number).columns
)
technical_allowed_prefixes = (
    'shape_', 'spacing_', 'fov_', 'voxel_volume_', 'finite_', 'zero_',
    'positive_fraction', 'positive_entropy_', 'foreground_', 'high_uptake_',
    'largest_component_', 'gradient_', 'slice_corr_', 'global_geometry_',
    'within_protocol_', 'technical_outlier_', 'correlation_',
    'metric_final_', 'registration_qc_', 'background_',
)
technical_columns = sorted(
    column for column in (qc_source_columns | registration_numeric)
    if column in data.columns
    and column not in {'is_pathologic'}
    and column.startswith(technical_allowed_prefixes)
    and not column.startswith('shape_active_')
)
biological_columns = [
    column for columns in biological_blocks.values() for column in columns
]

missing_fraction = data[biological_columns].replace(
    [np.inf, -np.inf], np.nan
).isna().mean()
biological_columns = missing_fraction[
    missing_fraction <= MAX_MISSING_FRACTION
].index.tolist()
biological_blocks = {
    name: [column for column in columns if column in biological_columns]
    for name, columns in biological_blocks.items()
}
if not biological_columns:
    raise ValueError('No quedaron features biológicas utilizables.')

display(pd.DataFrame({
    'bloque': [*biological_blocks.keys(), 'technical_qc', 'stability_qc'],
    'n_features': [
        *[len(columns) for columns in biological_blocks.values()],
        len(technical_columns), len(stability_columns),
    ],
    'rol': [
        *['predictor candidato'] * len(biological_blocks),
        'QC/dominio', 'QC de robustez',
    ],
}).style.hide(axis='index'))

## 2. Armonización por protocolo y PCA balanceado por bloque

Para cada variable se resta la mediana y se divide por IQR dentro de la familia de adquisición;
familias pequeñas o desconocidas usan estadísticos globales. Después se elimina redundancia
Spearman > 0,95 y cada bloque se divide por la raíz de su número de features, evitando que el
bloque más ancho domine el PCA.

In [ ]:
def fit_protocol_statistics(
    frame: pd.DataFrame,
    columns: list[str],
    groups: pd.Series,
    min_group_size: int = 5,
) -> dict[str, object]:
    numeric = frame[columns].replace([np.inf, -np.inf], np.nan).astype(float)
    global_median = numeric.median()
    global_iqr = (numeric.quantile(0.75) - numeric.quantile(0.25)).replace(0, 1)
    by_group: dict[str, tuple[pd.Series, pd.Series]] = {}
    for group in sorted(groups.astype(str).unique()):
        indices = groups.index[groups.astype(str) == group]
        if len(indices) < min_group_size:
            continue
        subset = numeric.loc[indices]
        median = subset.median().fillna(global_median)
        iqr = (
            subset.quantile(0.75) - subset.quantile(0.25)
        ).replace(0, np.nan).fillna(global_iqr)
        by_group[group] = (median, iqr)
    return {
        'columns': columns,
        'global_median': global_median,
        'global_iqr': global_iqr,
        'by_group': by_group,
    }


def apply_protocol_statistics(
    frame: pd.DataFrame,
    groups: pd.Series,
    statistics: dict[str, object],
) -> pd.DataFrame:
    columns = statistics['columns']
    numeric = frame[columns].replace([np.inf, -np.inf], np.nan).astype(float)
    result = pd.DataFrame(index=frame.index, columns=columns, dtype=float)
    global_median = statistics['global_median']
    global_iqr = statistics['global_iqr']
    for group in groups.astype(str).unique():
        indices = groups.index[groups.astype(str) == group]
        median, iqr = statistics['by_group'].get(
            str(group), (global_median, global_iqr)
        )
        result.loc[indices] = (
            numeric.loc[indices].fillna(median) - median
        ) / iqr
    return result.clip(-8, 8).astype(float)


def prune_correlated(
    frame: pd.DataFrame, columns: list[str], threshold: float
) -> tuple[list[str], list[str]]:
    kept: list[str] = []
    dropped: list[str] = []
    correlation = frame[columns].corr(method='spearman').abs().fillna(0)
    for column in columns:
        if any(correlation.loc[column, previous] > threshold for previous in kept):
            dropped.append(column)
        else:
            kept.append(column)
    return kept, dropped


full_statistics = fit_protocol_statistics(
    data, biological_columns, data['acquisition_family']
)
adjusted_biology = apply_protocol_statistics(
    data, data['acquisition_family'], full_statistics
)

retained_blocks: dict[str, list[str]] = {}
correlation_drops: list[dict[str, str]] = []
block_matrices: list[np.ndarray] = []
representation_names: list[str] = []
for block_name, block_columns in biological_blocks.items():
    if not block_columns:
        retained_blocks[block_name] = []
        continue
    kept, dropped = prune_correlated(
        adjusted_biology, block_columns, CORRELATION_THRESHOLD
    )
    retained_blocks[block_name] = kept
    correlation_drops.extend(
        {'block': block_name, 'dropped': column} for column in dropped
    )
    scaler = RobustScaler(quantile_range=(10, 90))
    block = scaler.fit_transform(adjusted_biology[kept])
    block = np.clip(block, -8, 8) / np.sqrt(max(len(kept), 1))
    block_matrices.append(block)
    representation_names.extend(kept)

X_biology = np.hstack(block_matrices)
n_components = min(12, X_biology.shape[0] - 1, X_biology.shape[1])
pca = PCA(n_components=n_components, random_state=RANDOM_SEED)
pca_scores = pca.fit_transform(X_biology)
embedding = data[['uid', 'is_pathologic', 'acquisition_family']].copy()
for index in range(pca_scores.shape[1]):
    embedding[f'PC{index + 1}'] = pca_scores[:, index]

loadings = pd.DataFrame(
    pca.components_.T,
    index=representation_names,
    columns=[f'PC{index + 1}' for index in range(n_components)],
)
top_loadings = pd.concat([
    loadings['PC1'].abs().nlargest(12).rename('abs_loading_PC1'),
    loadings['PC2'].abs().nlargest(12).rename('abs_loading_PC2'),
], axis=1).fillna(0)

figure, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(
    np.arange(1, n_components + 1),
    np.cumsum(pca.explained_variance_ratio_),
    marker='o',
)
axes[0].set(
    xlabel='Componentes', ylabel='Varianza acumulada',
    ylim=(0, 1.02), title='PCA biológico balanceado por bloque',
)
sns.scatterplot(
    data=embedding, x='PC1', y='PC2', hue='is_pathologic',
    alpha=0.75, ax=axes[1],
)
axes[1].set_title('PCA por etiqueta')
sns.scatterplot(
    data=embedding, x='PC1', y='PC2', hue='acquisition_family',
    legend=False, alpha=0.75, ax=axes[2],
)
axes[2].set_title('PCA por familia de adquisición')
plt.tight_layout()
plt.show()
display(Markdown('### Cargas principales'))
display(top_loadings.sort_values(['abs_loading_PC1', 'abs_loading_PC2'], ascending=False))
display(Markdown(
    f'**Features retenidas:** `{len(representation_names)}` · '
    f'**redundantes removidas:** `{len(correlation_drops)}`'
))

## 3. Proyección no lineal y vecinos biológicos

t-SNE se ajusta sobre PCA y, si la cohorte excede 1.500 casos, usa una muestra reproducible
sólo para visualización. Los vecinos usan el espacio biológico ajustado por protocolo y se
muestran desde los crops ya registrados del nodo 04, sin reabrir los NIfTI crudos.

In [ ]:
# Re-ejecutar esta celda no debe apilar widgets/figuras idénticos.
# El kernel conserva los objetos anteriores, por lo que cerramos el
# widget anterior y limpiamos la salida visual antes de redibujar.
if 'neighbor_output' in globals():
    neighbor_output.close()
clear_output(wait=True)
rng = np.random.default_rng(RANDOM_SEED)
saved_embedding = None
if RESUME and EMBEDDING_PATH.exists():
    candidate_embedding = pd.read_csv(
        EMBEDDING_PATH, dtype={'uid': 'string'}
    )
    if (
        len(candidate_embedding) == len(embedding)
        and set(candidate_embedding['uid']) == set(embedding['uid'])
        and {'NL1', 'NL2'} <= set(candidate_embedding.columns)
    ):
        saved_embedding = candidate_embedding.set_index('uid')
if saved_embedding is not None:
    embedding['NL1'] = embedding['uid'].map(saved_embedding['NL1'])
    embedding['NL2'] = embedding['uid'].map(saved_embedding['NL2'])
    embedding['projection_method'] = 't-SNE sobre PCA biológico'
    projection_indices = np.flatnonzero(embedding['NL1'].notna().to_numpy())
else:
    if len(data) > MAX_EMBEDDING_ROWS:
        projection_indices = np.sort(
            rng.choice(len(data), size=MAX_EMBEDDING_ROWS, replace=False)
        )
    else:
        projection_indices = np.arange(len(data))
    projection_input = pca_scores[
        projection_indices, :min(20, pca_scores.shape[1])
    ]
    perplexity = max(5, min(40, (len(projection_indices) - 1) // 10))
    tsne = TSNE(
        n_components=2, perplexity=perplexity, init='pca',
        learning_rate='auto', random_state=RANDOM_SEED,
        method='barnes_hut', angle=0.6, max_iter=TSNE_MAX_ITER,
        n_jobs=CPU_JOBS,
    )
    nonlinear = tsne.fit_transform(projection_input)
    embedding['NL1'] = np.nan
    embedding['NL2'] = np.nan
    embedding.loc[projection_indices, 'NL1'] = nonlinear[:, 0]
    embedding.loc[projection_indices, 'NL2'] = nonlinear[:, 1]
    embedding['projection_method'] = 't-SNE sobre PCA biológico'
    if SAVE_PRIVATE_OUTPUTS:
        atomic_csv(embedding, EMBEDDING_PATH)

plot_frame = embedding.loc[projection_indices].merge(
    data[['uid', 'spacing_x_mm', 'technical_outlier_score']],
    on='uid', how='left',
)
figure, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.scatterplot(
    data=plot_frame, x='NL1', y='NL2', hue='is_pathologic',
    alpha=0.75, ax=axes[0],
)
axes[0].set_title('t-SNE · etiqueta')
sns.scatterplot(
    data=plot_frame, x='NL1', y='NL2', hue='spacing_x_mm',
    palette='viridis', alpha=0.75, ax=axes[1],
)
axes[1].set_title('t-SNE · spacing')
sns.scatterplot(
    data=plot_frame, x='NL1', y='NL2', hue='technical_outlier_score',
    palette='magma', alpha=0.75, ax=axes[2],
)
axes[2].set_title('t-SNE · QC técnico')
plt.tight_layout()
plt.show()

neighbor_model = NearestNeighbors(
    metric='euclidean', n_neighbors=min(NEIGHBORS_TO_SHOW + 1, len(data))
)
neighbor_model.fit(X_biology)
distances, indices = neighbor_model.kneighbors(X_biology)
row_by_uid = {str(uid): index for index, uid in enumerate(data['uid'].astype(str))}


def load_mip(uid: str) -> np.ndarray:
    crop_path = CROPS_DIR / f'{uid}.npz'
    if not crop_path.exists():
        raise FileNotFoundError(
            f'Falta el crop registrado para {uid}: {crop_path}'
        )
    with np.load(crop_path, allow_pickle=False) as crop:
        volume = np.asarray(crop['volume_intensity01'], dtype=np.float32)
    values = volume[np.isfinite(volume)]
    low, high = np.percentile(values, [1, 99.5])
    volume01 = np.clip((volume - low) / max(high - low, 1e-8), 0, 1)
    return np.rot90(volume01.max(axis=0))


def show_neighbors(uid: str) -> None:
    query_index = row_by_uid[uid]
    neighbor_indices = indices[query_index]
    neighbor_distances = distances[query_index]
    figure, axes = plt.subplots(
        1, len(neighbor_indices), figsize=(3.2 * len(neighbor_indices), 3.5)
    )
    axes = np.atleast_1d(axes)
    rows = []
    for rank, (axis, neighbor_index, distance) in enumerate(
        zip(axes, neighbor_indices, neighbor_distances)
    ):
        neighbor_uid = str(data.iloc[neighbor_index]['uid'])
        label = int(data.iloc[neighbor_index]['is_pathologic'])
        family = str(data.iloc[neighbor_index]['acquisition_family'])
        axis.imshow(load_mip(neighbor_uid), cmap='hot', vmin=0, vmax=1)
        axis.set_title(
            f'{rank}. {neighbor_uid}\ny={label} · {family} · d={distance:.2f}'
        )
        axis.axis('off')
        rows.append({
            'rank': rank, 'uid': neighbor_uid, 'is_pathologic': label,
            'acquisition_family': family, 'distance': distance,
        })
    plt.tight_layout()
    plt.show()
    display(pd.DataFrame(rows).style.hide(axis='index'))


uid_selector = widgets.Dropdown(
    options=sorted(row_by_uid), value=sorted(row_by_uid)[0], description='UID:',
    layout=widgets.Layout(width='420px'),
    style={'description_width': '60px'},
)
neighbor_output = widgets.interactive_output(show_neighbors, {'uid': uid_selector})
display(widgets.VBox([uid_selector, neighbor_output]))

## 3.1 Comparación robusta t-SNE / UMAP

La búsqueda varía los hiperparámetros de ambos métodos y repite cada ensayo con tres
semillas. Se reportan dos selecciones: **estructura**, que prioriza fidelidad y estabilidad
penalizando agrupamiento por adquisición/spacing, y **balanceada**, que además considera la
separación de etiquetas. Las etiquetas sólo evalúan una proyección ya ajustada: nunca se
entregan a t-SNE ni a UMAP. Por ello la selección balanceada sigue siendo exploratoria y no
reemplaza las métricas OOF del clasificador.

In [ ]:
baseline_coordinates = embedding[['NL1', 'NL2']].to_numpy(dtype=float)
if not np.isfinite(baseline_coordinates).all():
    baseline_coordinates = None

embedding_search_result = run_embedding_search(
    pca_scores,
    y=data['is_pathologic'].to_numpy(dtype=int),
    groups=data['acquisition_family'].astype(str).to_numpy(),
    spacing=data['spacing_x_mm'].to_numpy(dtype=float),
    uids=data['uid'].astype(str).to_numpy(),
    output_dir=EMBEDDING_SEARCH_DIR,
    experiment_config={
        'node5_run_id': NODE5_RUN_ID,
        'embedding_run_id': EMBEDDING_RUN_ID,
        'source': str(EMBEDDING_PATH),
        'pca_columns': [
            f'PC{index + 1}' for index in range(pca_scores.shape[1])
        ],
    },
    n_trials_per_method=EMBEDDING_TRIALS_PER_METHOD,
    seeds=EMBEDDING_SEEDS,
    cpu_jobs=CPU_JOBS,
    evaluation_rows=min(800, len(data)),
    evaluation_neighbors=15,
    baseline_coordinates=baseline_coordinates,
)
comparison_columns = [
    'method', 'selection', 'label_silhouette',
    'label_knn_balanced_accuracy', 'neighborhood_trustworthiness',
    'seed_neighborhood_stability', 'acquisition_family_confound',
    'spacing_confound', 'balanced_score',
    'labels_used_to_select_trial',
]
display(Markdown('### Comparación cuantitativa de proyecciones'))
display(
    embedding_search_result.comparison[comparison_columns]
    .style.format({
        'label_silhouette': '{:.3f}',
        'label_knn_balanced_accuracy': '{:.3f}',
        'neighborhood_trustworthiness': '{:.3f}',
        'seed_neighborhood_stability': '{:.3f}',
        'acquisition_family_confound': '{:.3f}',
        'spacing_confound': '{:.3f}',
        'balanced_score': '{:.3f}',
    })
    .hide(axis='index')
)

optimized_plot = embedding_search_result.coordinates.merge(
    data[['uid', 'technical_outlier_score']],
    on='uid', how='left', validate='one_to_one',
)
figure, axes = plt.subplots(2, 3, figsize=(18, 10))
for row, method in enumerate(('tsne', 'umap')):
    x_column = f'{method}_balanced_1'
    y_column = f'{method}_balanced_2'
    sns.scatterplot(
        data=optimized_plot, x=x_column, y=y_column,
        hue='is_pathologic', alpha=0.72, s=25, ax=axes[row, 0],
    )
    axes[row, 0].set_title(f'{method.upper()} optimizado · etiqueta')
    sns.scatterplot(
        data=optimized_plot, x=x_column, y=y_column,
        hue='spacing_x_mm', palette='viridis',
        alpha=0.72, s=25, ax=axes[row, 1],
    )
    axes[row, 1].set_title(f'{method.upper()} optimizado · spacing')
    sns.scatterplot(
        data=optimized_plot, x=x_column, y=y_column,
        hue='technical_outlier_score', palette='magma',
        alpha=0.72, s=25, ax=axes[row, 2],
    )
    axes[row, 2].set_title(f'{method.upper()} optimizado · QC técnico')
plt.tight_layout()
plt.show()

display(Markdown(
    '**Lectura correcta:** una mejora sólo es convincente si aumenta la separación '
    'sin perder trustworthiness/estabilidad ni aumentar la confusión por familia o '
    'spacing. La proyección balanceada fue seleccionada mirando la etiqueta y, por '
    'tanto, no constituye validación fuera de muestra.'
))

## 4. Anomalías separadas por rol

`technical_anomaly` se ajusta sólo con QC/adquisición. `biological_anomaly` combina Isolation
Forest, error de reconstrucción PCA y Mahalanobis regularizada en el espacio biológico. Sus
percentiles priorizan revisión; no se usan como features del modelo.

In [ ]:
technical_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler(quantile_range=(10, 90))),
])
X_technical = technical_preprocessor.fit_transform(
    data[technical_columns].replace([np.inf, -np.inf], np.nan)
)
technical_isolation = IsolationForest(
    n_estimators=ISOLATION_TREES, max_samples=min(512, len(data)),
    contamination='auto', random_state=RANDOM_SEED,
    n_jobs=CPU_JOBS,
)
technical_score = -technical_isolation.fit(X_technical).score_samples(X_technical)

biological_isolation = IsolationForest(
    n_estimators=ISOLATION_TREES, max_samples=min(512, len(data)),
    contamination='auto', random_state=RANDOM_SEED,
    n_jobs=CPU_JOBS,
)
biological_isolation_score = -biological_isolation.fit(X_biology).score_samples(
    X_biology
)
cumulative = np.cumsum(pca.explained_variance_ratio_)
components_90 = min(
    int(np.searchsorted(cumulative, 0.90) + 1), pca_scores.shape[1]
)
keep_components = max(1, min(components_90, pca_scores.shape[1] - 1))
reduced_scores = np.zeros_like(pca_scores)
reduced_scores[:, :keep_components] = pca_scores[:, :keep_components]
reconstructed = pca.inverse_transform(reduced_scores)
pca_error = np.mean((X_biology - reconstructed) ** 2, axis=1)
covariance_components = min(8, pca_scores.shape[1], max(1, len(data) - 2))
covariance = LedoitWolf().fit(pca_scores[:, :covariance_components])
mahalanobis = covariance.mahalanobis(pca_scores[:, :covariance_components])

outliers = data[[
    'uid', 'is_pathologic', 'acquisition_family', 'technical_outlier_score',
    'feature_available', 'background_qc_valid', 'background_floor_applied',
]].copy()
outliers['technical_isolation_score'] = technical_score
outliers['biological_isolation_score'] = biological_isolation_score
outliers['pca_reconstruction_error'] = pca_error
outliers['biological_mahalanobis'] = mahalanobis
outliers['registration_qc_flag'] = data['registration_qc_flag'].fillna(True).astype(bool)
outliers['technical_anomaly_rank'] = outliers[
    'technical_isolation_score'
].rank(pct=True)
biological_ranks = []
for column in [
    'biological_isolation_score', 'pca_reconstruction_error',
    'biological_mahalanobis',
]:
    rank_column = f'{column}_rank'
    outliers[rank_column] = outliers[column].rank(pct=True)
    biological_ranks.append(rank_column)
outliers['biological_anomaly_rank'] = outliers[biological_ranks].mean(axis=1)
outliers['registration_risk_rank'] = (
    1 - data['correlation_after'].rank(pct=True)
).fillna(1.0)
outliers['background_risk'] = (
    (~outliers['background_qc_valid'])
    | outliers['background_floor_applied']
).astype(float)
outliers['review_score'] = outliers[[
    'technical_anomaly_rank', 'biological_anomaly_rank',
    'registration_risk_rank', 'background_risk',
]].mean(axis=1)
outliers = outliers.sort_values('review_score', ascending=False)
if SAVE_PRIVATE_OUTPUTS:
    atomic_csv(outliers, OUTLIER_PATH)
display(outliers.head(25).style.format({
    'technical_anomaly_rank': '{:.1%}',
    'biological_anomaly_rank': '{:.1%}',
    'registration_risk_rank': '{:.1%}',
    'review_score': '{:.1%}',
}).hide(axis='index'))

## 5. Validación agrupada y optimización bayesiana

Los resultados previos fijan `radiomics_only` (forma + textura) como representación candidata:
superó a las variables técnicas, mientras semiquantificación y first-order no pasan el gate de
fondo. La armonización continúa ajustándose sólo con el fold de entrenamiento. Optuna ejecuta
50 ensayos completos para regresión logística, random forest y XGBoost; cada estudio y cada
fold OOF persisten por separado, por lo que una interrupción no obliga a repetirlos.

In [ ]:
y = data['is_pathologic'].to_numpy()
groups = data['acquisition_family'].astype(str).to_numpy()
unique_groups = np.unique(groups)
n_splits = min(5, len(unique_groups), int(pd.Series(y).value_counts().min()))
split_method = 'StratifiedGroupKFold por acquisition_family'
if n_splits >= 2:
    splitter = StratifiedGroupKFold(
        n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED
    )
    candidate_splits = list(splitter.split(data, y, groups))
    valid_group_splits = all(len(np.unique(y[train])) == 2 for train, _ in candidate_splits)
else:
    valid_group_splits = False
if not valid_group_splits:
    n_splits = min(5, int(pd.Series(y).value_counts().min()))
    if n_splits < 2:
        raise ValueError('No hay suficientes casos por clase para validación cruzada.')
    splitter = StratifiedKFold(
        n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED
    )
    candidate_splits = list(splitter.split(data, y))
    split_method = 'StratifiedKFold fallback; revisar grupos insuficientes'

radiomics_columns = retained_blocks['shape'] + retained_blocks['texture']
semiquant_columns = retained_blocks['semiquant']
biomarkers_all_columns = [
    column for columns in retained_blocks.values() for column in columns
]
feature_sets = {
    'technical_only': {'biology': [], 'technical': technical_columns},
    'semiquant_only': {'biology': semiquant_columns, 'technical': []},
    'radiomics_only': {'biology': radiomics_columns, 'technical': []},
    'biomarkers_all': {'biology': biomarkers_all_columns, 'technical': []},
    'biomarkers_plus_technical': {
        'biology': biomarkers_all_columns, 'technical': technical_columns,
    },
}


def fold_matrix(
    train_indices: np.ndarray,
    test_indices: np.ndarray,
    biology_columns: list[str],
    technical_columns_fold: list[str],
) -> tuple[np.ndarray, np.ndarray]:
    train_parts: list[np.ndarray] = []
    test_parts: list[np.ndarray] = []
    if biology_columns:
        train_frame = data.iloc[train_indices]
        test_frame = data.iloc[test_indices]
        statistics = fit_protocol_statistics(
            train_frame,
            biology_columns,
            train_frame['acquisition_family'],
        )
        train_biology = apply_protocol_statistics(
            train_frame, train_frame['acquisition_family'], statistics
        )[biology_columns]
        test_biology = apply_protocol_statistics(
            test_frame, test_frame['acquisition_family'], statistics
        )[biology_columns]
        scaler = RobustScaler(quantile_range=(10, 90))
        train_parts.append(scaler.fit_transform(train_biology))
        test_parts.append(scaler.transform(test_biology))
    if technical_columns_fold:
        imputer = SimpleImputer(strategy='median')
        scaler = RobustScaler(quantile_range=(10, 90))
        train_technical = imputer.fit_transform(
            data.iloc[train_indices][technical_columns_fold].replace(
                [np.inf, -np.inf], np.nan
            )
        )
        test_technical = imputer.transform(
            data.iloc[test_indices][technical_columns_fold].replace(
                [np.inf, -np.inf], np.nan
            )
        )
        train_parts.append(scaler.fit_transform(train_technical))
        test_parts.append(scaler.transform(test_technical))
    return np.hstack(train_parts), np.hstack(test_parts)


if OPTUNA_FEATURE_SET not in feature_sets:
    raise ValueError(f'Feature set Optuna desconocido: {OPTUNA_FEATURE_SET}')
selected_roles = feature_sets[OPTUNA_FEATURE_SET]
if not selected_roles['biology'] and not selected_roles['technical']:
    raise ValueError(f'Feature set vacío: {OPTUNA_FEATURE_SET}')

optuna_folds: list[FoldData] = []
fold_id = np.full(len(data), -1, dtype=int)
for fold, (train_indices, test_indices) in enumerate(candidate_splits):
    X_train, X_test = fold_matrix(
        train_indices,
        test_indices,
        selected_roles['biology'],
        selected_roles['technical'],
    )
    optuna_folds.append(FoldData(
        fold=fold,
        train_indices=np.asarray(train_indices, dtype=int),
        valid_indices=np.asarray(test_indices, dtype=int),
        X_train=np.asarray(X_train, dtype=np.float32),
        X_valid=np.asarray(X_test, dtype=np.float32),
        y_train=np.asarray(y[train_indices], dtype=int),
        y_valid=np.asarray(y[test_indices], dtype=int),
    ))
    fold_id[test_indices] = fold

optuna_result = run_optuna_experiment(
    optuna_folds,
    uids=data['uid'].astype(str).tolist(),
    y=y,
    groups=groups,
    output_dir=OPTUNA_OUTPUT_DIR,
    experiment_config={
        'node5_config_hash': node5_hash,
        'feature_set': OPTUNA_FEATURE_SET,
        'feature_names': (
            selected_roles['biology'] + selected_roles['technical']
        ),
        'split_method': split_method,
        'n_splits': n_splits,
    },
    n_trials_per_model=OPTUNA_TRIALS_PER_MODEL,
    models=OPTUNA_MODELS,
    random_seed=RANDOM_SEED,
    cpu_jobs=CPU_JOBS,
    prefer_gpu=PREFER_XGBOOST_GPU,
)
optuna_oof = optuna_result.oof_predictions.merge(
    pd.DataFrame({'uid': data['uid'].astype(str), 'fold': fold_id}),
    on='uid', how='left', validate='one_to_one',
)
optuna_metrics = optuna_result.metrics.assign(
    feature_set=OPTUNA_FEATURE_SET,
    n_features=(
        len(selected_roles['biology']) + len(selected_roles['technical'])
    ),
    split_method=split_method,
)
display(Markdown(
    f'**Validación:** `{split_method}` · **folds:** `{n_splits}` · '
    f'**XGBoost:** `{optuna_result.xgboost_device}`'
))
display(Markdown('### Mejor ensayo Optuna por clasificador'))
display(optuna_result.tuning_summary.style.format({
    'best_cv_log_loss': '{:.4f}',
}).hide(axis='index'))
display(Markdown('### Predicciones OOF con los hiperparámetros seleccionados'))
display(optuna_metrics.style.format({
    'oof_auc': '{:.3f}', 'oof_log_loss': '{:.3f}',
    'oof_brier': '{:.3f}', 'oof_balanced_accuracy_0_5': '{:.3f}',
}).hide(axis='index'))
display(Markdown(
    '> Estas métricas son exploratorias: los mismos folds participan en la '
    'selección de hiperparámetros. La estimación final requiere validación anidada '
    'o un holdout intacto.'
))

best_classifier = str(
    optuna_metrics.nsmallest(1, 'oof_log_loss').iloc[0]['model']
)
selected_probability_column = (
    'p_ensemble_oof' if best_classifier == 'mean_ensemble'
    else f'p_{best_classifier}_oof'
)
p_mean = optuna_oof[selected_probability_column].to_numpy(dtype=float)
model_probability_columns = [
    f'p_{model_name}_oof' for model_name in OPTUNA_MODELS
]
observed_probability = np.where(y == 1, p_mean, 1 - p_mean)
label_audit = data[[
    'uid', 'is_pathologic', 'acquisition_family',
    'registration_qc_flag', 'correlation_after',
    'feature_available', 'background_qc_valid',
    'background_floor_applied',
]].copy()
label_audit['selected_feature_set'] = OPTUNA_FEATURE_SET
label_audit['selected_classifier'] = best_classifier
for probability_column in model_probability_columns:
    label_audit[probability_column] = optuna_oof[
        probability_column
    ].to_numpy(dtype=float)
label_audit['p_mean_oof'] = p_mean
label_audit['predictive_entropy'] = -(
    p_mean * np.log(p_mean) + (1 - p_mean) * np.log(1 - p_mean)
)
label_audit['model_disagreement'] = optuna_oof[
    model_probability_columns
].std(axis=1).to_numpy(dtype=float)
label_audit['label_surprise'] = -np.log(observed_probability)
label_audit = label_audit.merge(
    outliers[['uid', 'technical_anomaly_rank', 'biological_anomaly_rank', 'review_score']],
    on='uid', how='left', validate='one_to_one',
)
label_audit['expert_review_priority'] = (
    label_audit['label_surprise'].rank(pct=True)
    + label_audit['model_disagreement'].rank(pct=True)
    + label_audit['review_score'].rank(pct=True)
) / 3
label_audit = label_audit.sort_values('expert_review_priority', ascending=False)
display(Markdown(
    f'### Revisión experta · `{OPTUNA_FEATURE_SET}` + `{best_classifier}`'
))
display(label_audit.head(25).style.format({
    **{column: '{:.3f}' for column in model_probability_columns},
    'p_mean_oof': '{:.3f}', 'expert_review_priority': '{:.1%}',
}).hide(axis='index'))

## 6. Diagnóstico por protocolo y persistencia

Las métricas OOF son evidencia exploratoria del pipeline tabular, no una estimación final del
modelo de imágenes. La promoción requiere repetir exactamente el mismo contrato sobre folds
externos/holdout y agregar la rama de crops 3D mediante una ablación separada.

In [ ]:
selected_oof = optuna_oof[[
    'uid', 'is_pathologic', 'acquisition_family', 'fold'
]].copy()
selected_oof['p_mean_oof'] = p_mean
protocol_diagnostics = (
    selected_oof.groupby('acquisition_family')
    .agg(
        n=('uid', 'size'),
        pathologic_rate=('is_pathologic', 'mean'),
        mean_probability=('p_mean_oof', 'mean'),
        mean_absolute_error=(
            'p_mean_oof',
            lambda values: float(np.mean(np.abs(
                values.to_numpy()
                - selected_oof.loc[values.index, 'is_pathologic'].to_numpy()
            ))),
        ),
    )
    .reset_index()
    .sort_values('n', ascending=False)
)
display(Markdown('### Diagnóstico por familia de adquisición'))
display(protocol_diagnostics.head(30).style.hide(axis='index'))

if SAVE_PRIVATE_OUTPUTS:
    atomic_csv(embedding, EMBEDDING_PATH)
    atomic_csv(outliers, OUTLIER_PATH)
    atomic_csv(label_audit, LABEL_AUDIT_PATH)
    atomic_csv(optuna_oof, OOF_PATH)
    atomic_csv(optuna_metrics, METRICS_PATH)
    atomic_csv(protocol_diagnostics, PROTOCOL_DIAGNOSTICS_PATH)
    embedding_config_path = NODE5_OUTPUT_DIR / 'embedding_full_config.json'
    temporary_embedding_config = embedding_config_path.with_suffix('.json.tmp')
    temporary_embedding_config.write_text(
        json.dumps({
            'algorithm_version': 'blockwise_optuna_v1',
            'node4_profile': NODE4_PROFILE,
            'node5_run_id': NODE5_RUN_ID,
            'cpu_jobs': CPU_JOBS,
            'isolation_trees': ISOLATION_TREES,
            'tsne_max_iter': TSNE_MAX_ITER,
            'embedding_run_id': EMBEDDING_RUN_ID,
            'embedding_trials_per_method': EMBEDDING_TRIALS_PER_METHOD,
            'embedding_search_dir': str(EMBEDDING_SEARCH_DIR),
            'embedding_labels_used_for_fit': False,
            'embedding_balanced_selection_uses_labels': True,
            'neighbor_images_source': str(CROPS_DIR),
            'background_blocks_enabled': background_blocks_enabled,
            'feature_coverage': feature_coverage,
            'background_valid_fraction': background_valid_fraction,
            'biological_blocks_available': biological_blocks,
            'retained_blocks_after_correlation_filter': retained_blocks,
            'correlation_threshold': CORRELATION_THRESHOLD,
            'protocol_adjustment': 'training-family median and IQR; global fallback',
            'block_weighting': 'each block divided by sqrt(number of retained features)',
            'pca_components': n_components,
            'pca_components_for_90_percent': components_90,
            'split_method': split_method,
            'n_splits': n_splits,
            'optuna_feature_set': OPTUNA_FEATURE_SET,
            'best_classifier_exploratory': best_classifier,
            'optuna_trials_per_model': OPTUNA_TRIALS_PER_MODEL,
            'optuna_models': OPTUNA_MODELS,
            'xgboost_device': optuna_result.xgboost_device,
            'anomaly_scores_are_not_predictors': True,
            'random_seed': RANDOM_SEED,
        }, indent=2),
        encoding='utf-8',
    )
    replace_with_retry(temporary_embedding_config, embedding_config_path)
    display(Markdown(
        f'Artefactos del nodo 05 guardados en `{NODE5_OUTPUT_DIR}`. '
        f'El nodo 04 permanece intacto en `{NODE4_OUTPUT_DIR}`.'
    ))